# FedMelanoma — EfficientNet-B0 + Hybrid FedProx/FedNova (Colab)

Runs the **real federated training loop** (`experiments/effnet_b0_fedprox_fednova/train_fedprox_fednova.py`, unmodified): 10 hospitals, FedProx local training on each client, FedNova aggregation on the server, real gradients and real per-hospital image data — not the metadata-only simulation in `server/simulation/`.

Reproduces: Springer ICACECS 2025, "Federated Learning for Skin Cancer Detection with Optimized Client Selection and Hybrid Aggregation Strategies" (~93.56% reported accuracy), using full participation instead of their unpublished client-selection formula — see the experiment's `README.md` for the honest gap analysis.

**Before running:**
1. Runtime → Change runtime type → GPU (T4 is fine).
2. Runtime → Run all. One Drive-auth popup to approve (used only for checkpoint/log backups), then it runs unattended — the dataset itself downloads straight from a public Drive link, no manual upload needed. A short smoke test runs first to catch setup problems early, then the full federated run.

Checkpoint + JSON log write **directly to your Drive** every round (the script's own `--checkpoint-path` / `--log-path` point at Drive), so a disconnect mid-run loses at most the in-progress round.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- Config ----
REPO_URL = "https://github.com/rudraverma46/Fedmelanoma"
REPO_BRANCH = "feature/effnet-b0-fedprox-fednova"

# Public Google Drive file containing the hospital_partitions dataset (zipped).
GDRIVE_FILE_ID = "19_xX8Kkb5rNK4r4qSGFS64Yw8WslYwn3"

DRIVE_RUNS_DIR = "/content/drive/MyDrive/FedMelanoma_runs"
LOCAL_ROOT = "/content"

# Federated run hyperparameters (README.md defaults; override as needed)
ROUNDS = 10
LOCAL_EPOCHS = 2
MU = 0.01           # FedProx proximal coefficient
BATCH_SIZE = 16
LR = 1e-4
WEIGHT_DECAY = 1e-4

In [ ]:
import os
os.chdir(LOCAL_ROOT)
!rm -rf Fedmelanoma
!git clone --branch "{REPO_BRANCH}" --single-branch {REPO_URL} Fedmelanoma

In [ ]:
# torch/torchvision are already installed (CUDA-matched) in Colab — leave them alone.
!pip install -q timm opencv-python-headless gdown

In [ ]:
import subprocess
import time

ZIP_PATH = os.path.join(LOCAL_ROOT, "hospital_partitions.zip")
EXTRACT_DIR = os.path.join(LOCAL_ROOT, "dataset_extracted")

os.chdir(LOCAL_ROOT)

if not os.path.exists(ZIP_PATH):
    print("Downloading dataset from Google Drive...")
    t0 = time.time()
    !gdown --id {GDRIVE_FILE_ID} -O {ZIP_PATH}
    print(f"Downloaded in {time.time()-t0:.0f}s.")

if not os.path.isdir(EXTRACT_DIR):
    print("Extracting...")
    t0 = time.time()
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d "{EXTRACT_DIR}"
    print(f"Extracted in {time.time()-t0:.0f}s.")

# Auto-detect where the 10 hospital subfolders actually landed (handles any
# wrapping folder the zip may or may not have).
result = subprocess.run(
    ["find", EXTRACT_DIR, "-maxdepth", "4", "-type", "d", "-iname", "CityGeneral_Multispecialty_Hospital"],
    capture_output=True, text=True,
)
matches = [line for line in result.stdout.splitlines() if line]
assert matches, (
    f"Could not find a 'CityGeneral_Multispecialty_Hospital' folder under {EXTRACT_DIR} "
    "after extraction — the zip's internal layout doesn't match what train_fedprox_fednova.py expects. "
    f"Contents: {os.listdir(EXTRACT_DIR)}"
)
DATA_ROOT = os.path.dirname(matches[0])
print(f"DATA_ROOT = {DATA_ROOT}")
!ls "{DATA_ROOT}"

In [ ]:
os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)

CKPT_PATH = os.path.join(DRIVE_RUNS_DIR, "best_effnet_b0_fedprox_fednova.pth")
LOG_PATH = os.path.join(DRIVE_RUNS_DIR, "fedprox_fednova_training_log.json")

EXPERIMENT_DIR = os.path.join(LOCAL_ROOT, "Fedmelanoma", "experiments", "effnet_b0_fedprox_fednova")
os.chdir(EXPERIMENT_DIR)

print(f"Checkpoint -> {CKPT_PATH}")
print(f"Log        -> {LOG_PATH}")

## Smoke test (recommended, ~1 minute)

Tiny data subset, not meaningful accuracy — just confirms data loading, FedProx local training, and FedNova aggregation run end-to-end before committing to the full run below. Safe to skip if you've already verified this.

In [ ]:
!python train_fedprox_fednova.py \
    --data-root "{DATA_ROOT}" \
    --rounds 2 --local-epochs 1 --limit-per-hospital 12 \
    --checkpoint-path /content/smoke_test.pth \
    --log-path /content/smoke_test_log.json

## Full federated run

In [ ]:
# Real federated run — all 10 hospitals, every round (full participation).
# Rough expectation from the README's compute note: several hours on CPU;
# a T4 here should be a small fraction of that, but this hasn't been
# benchmarked on GPU, so treat any ETA as a guess and just let it run.
!python train_fedprox_fednova.py \
    --data-root "{DATA_ROOT}" \
    --rounds {ROUNDS} \
    --local-epochs {LOCAL_EPOCHS} \
    --mu {MU} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --weight-decay {WEIGHT_DECAY} \
    --num-workers 2 \
    --checkpoint-path "{CKPT_PATH}" \
    --log-path "{LOG_PATH}"